# Grover’s Search: Minimal Demonstration $(N=4)$

## Objectives
- Implement Grover’s search for a 2-qubit database $(N=4)$ with a single marked item.
- Verify quadratic speed-up signal via success probability vs. iteration count.

## Setup
```python
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
```


## Theory Snapshot
Grover’s algorithm amplifies the amplitude of marked states using repeated applications of the **oracle** and **diffuser**. For **one marked item** among $N$ items, the optimal number of iterations is $\approx [\pi/\sqrt(N/4)]$.

## Circuit / Model
We use two qubits $(N=4)$ and mark the state $|11\rangle$ as the target. The oracle applies a phase flip to $|11\rangle$; the diffuser reflects about the uniform superposition.

In [ ]:
from qiskit import QuantumCircuit

def oracle_mark_11():
    qc = QuantumCircuit(2)
    qc.cz(0,1)  # phase flip on |11>
    return qc

def diffuser():
    qc = QuantumCircuit(2)
    qc.h([0,1]); qc.x([0,1]); qc.cz(0,1); qc.x([0,1]); qc.h([0,1])
    return qc

def grover_iteration():
    qc = QuantumCircuit(2)
    qc.compose(oracle_mark_11(), inplace=True)
    qc.compose(diffuser(), inplace=True)
    return qc

## Experiments
We measure the probability of the marked state as a function of the number of Grover iterations $k \in {0,1,2}$. For $N=4$, $k=1$ is optimal.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def run_grover(k: int):
    qc = QuantumCircuit(2)
    qc.h([0,1])  # uniform superposition
    for _ in range(k):
        qc.compose(grover_iteration(), inplace=True)
    
    # Exact state and probability of |11> (index 3 in little-endian)
    sv = Statevector.from_instruction(qc)
    p11 = float(sv.probabilities()[3])   # 3 == int('11', 2)
    return p11

probs = [run_grover(k) for k in [0,1,2]]
print("P(|11>) for k=0,1,2:", probs)  # -> [0.25, 1.0, 0.25]


## Metrics & Plots
We plot the success probability for the marked item $|11\rangle$ as $k$ varies.

In [ ]:
try:
    import matplotlib.pyplot as plt
    ks = [0,1,2]
    plt.figure(); plt.plot(ks, probs, marker='o'); plt.xticks(ks)
    plt.xlabel('Grover iterations (k)'); plt.ylabel('P(success on |11>)')
    plt.title('Grover Success Probability vs Iterations (N=4)'); plt.grid(True)
except Exception as e:
    print('Plot skipped:', e)

## Results & Discussion
- For $N=4$, a single iteration $(k=1)$ maximizes success probability as expected from theory.
- Over-iteration $(k=2)$ rotates the state past the optimum, reducing success probability.
- With noise models or real backends, the optimal $k$ may shift and overall success decreases.

## References
- Grover, *A Fast Quantum Mechanical Algorithm for Database Search* (1996)
- IBM Qiskit Textbook — Grover’s Algorithm